# Exploration de données

La SNCF a publié une mise à jour de son fichier de comptage de montants, mais la mise à jour ne reprend pas toutes les données de la précédente version. Ce notebook essaye d'illustrer la démarche de fusion de fichiers.

**WIP** : ce notebook est en cours de réalisation et n'est pas destiné à une présentation.

## Chargement des données

In [ ]:
from pathlib import Path
file_old_path = Path('__file__').resolve().parent.parent / 'assets' / 'comptage-voyageurs-trains-transilien-old.csv'
file_new_path = Path('__file__').resolve().parent.parent / 'assets' / 'comptage-voyageurs-trains-transilien-210716.csv'

print(file_old_path.exists())
print(file_new_path.exists())

import pandas as pd

Création des dataframes et renommage d'une colonne pour uniformisation.

In [ ]:
train_old_data = pd.read_csv(file_old_path, sep=";")
train_new_data = pd.read_csv(file_new_path, sep=";")
train_new_data = train_new_data.rename(columns={'Année': "Year"})

Affichage des informations de base pour vérification.

In [ ]:
train_old_data.info()

In [ ]:

train_new_data.info()

## Fusion des dataframes
Fusion et affichage des informations de base.

In [ ]:
train_data = pd.concat([train_old_data, train_new_data], ignore_index=True)
train_data.info()

## Suppression des doublons
Basé sur le code gare, date, ligne et tranche horaire.

In [ ]:
train_data.drop_duplicates(subset=['Code Gare', 'Date de comptage', 'Ligne', 'Tranche horaire'], inplace=True)
train_data.info()

In [ ]:
train_data[['Nom gare', 'Ligne']].value_counts()

In [ ]:
train_data.loc[train_data['Nom gare'] == "AEROPORT CHARLES DE GAULLE 2 TGV"].sort_values(by="Montants", ascending=False)

In [ ]:
pd.merge(train_new_data, train_old_data, on=['Code Gare', 'Date de comptage', 'Ligne', 'Tranche horaire'])

In [ ]:
df1_avec_marqueur = train_old_data.merge(
    train_new_data.drop_duplicates(),
    on=['Code Gare', 'Date de comptage', 'Ligne', 'Tranche horaire'],
    how='left',
    indicator=True
)

# Voir combien de lignes sont en double
print(df1_avec_marqueur['_merge'].value_counts())

In [ ]:
doublons = df1_avec_marqueur[df1_avec_marqueur['_merge'] == 'both']
print(f"Nombre de doublons : {len(doublons)}")
doublons